# Databricks Notebook: 02_process_countries_data.ipynb

# Este notebook processa os dados brutos de países e os salva como uma tabela Delta no DBFS.

In [0]:
import pyspark.sql.functions as F

In [0]:
import logging
from typing import Dict, List, Optional, Any
from pyspark.sql import SparkSession, DataFrame
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, ArrayType, LongType,
    DoubleType, MapType
)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class ProcessingError(Exception):
    """Custom exception for data processing errors."""
    pass

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a SparkSession with Delta Lake support.

    Args:
        app_name (str): The name of the Spark application.

    Returns:
        SparkSession: The configured SparkSession.
    """
    logger.info(f"Creating SparkSession for application: {app_name}")
    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()
    logger.info("SparkSession created successfully.")
    return spark

def define_schema() -> StructType:
    """
    Defines the schema for the raw country data.

    Returns:
        StructType: The defined Spark DataFrame schema.
    """
    return StructType([
        StructField("name", StructType([
            StructField("common", StringType(), True),
            StructField("official", StringType(), True)
        ]), True),
        StructField("cca2", StringType(), True),
        StructField("region", StringType(), True),
        StructField("subregion", StringType(), True),
        StructField("population", LongType(), True),
        StructField("area", DoubleType(), True),
        StructField("capital", ArrayType(StringType()), True),
        StructField("currencies", MapType(StringType(), StructType([
            StructField("name", StringType(), True),
            StructField("symbol", StringType(), True)
        ])), True),
        StructField("languages", MapType(StringType(), StringType()), True),
        StructField("latlng", ArrayType(DoubleType()), True),
        StructField("timezones", ArrayType(StringType()), True),
        StructField("borders", ArrayType(StringType()), True),
    ])

def process_countries_data(spark: SparkSession, input_path: str, output_path: str) -> None:
    """
    Reads raw country data, processes it, and writes to Delta Lake.

    Args:
        spark (SparkSession): The active SparkSession.
        input_path (str): Path to the raw JSON data on DBFS.
        output_path (str): Path to save the processed Delta table on DBFS.
    """
    logger.info(f"Reading raw data from {input_path}")
    try:
        # Read JSON data with the defined schema
        df = spark.read.schema(define_schema()).json(input_path)
        logger.info("Raw data read successfully. Starting processing...")

        # Flatten and process data
        processed_df = df.withColumn("country_name", F.col("name.common")) \
            .withColumn("capital", F.col("capital")[0]) \
            .withColumn("currency_code", F.element_at(F.map_keys(F.col("currencies")), 1)) \
            .withColumn("currency_name", F.col("currencies")[F.col("currency_code")].getItem("name")) \
            .withColumn("languages", F.array_distinct(F.map_values(F.col("languages")))) \
            .withColumn("population_density", F.col("population") / F.col("area")) \
            .select(
                "country_name",
                "cca2",
                "region",
                "subregion",
                "population",
                "area",
                "capital",
                "currency_code",
                "currency_name",
                "languages",
                "latlng",
                "borders",
                "population_density"
            )

        logger.info(f"Writing processed data to Delta Lake at {output_path}")
        processed_df.write.format("delta").mode("overwrite").save(output_path)
        logger.info("Data successfully processed and saved to Delta Lake.")

    except Exception as e:
        logger.error(f"Error processing data: {e}")
        raise ProcessingError(f"Failed to process data: {e}")

# Recebe o caminho do arquivo de entrada do notebook anterior via dbutils.widgets.get
input_path = "/Volumes/workspace/default/data"
output_path = "/Volumes/workspace/default/data/processed/processed_countries"

spark = create_spark_session("CountryDataProcessing")
process_countries_data(spark, input_path, output_path)
spark.stop()

# Retorna o caminho de saída para o notebook orquestrador
dbutils.notebook.exit(output_path)
